# 🧪 Qwen2.5-7B Instruct — Quantization Quality Benchmark
**Goal:** Compare your Q4_K_M quantized model (built with llama.cpp) against the original BF16 model from HuggingFace.

**What this measures:** Quality/intelligence loss only — perplexity, reasoning, knowledge, math, and hallucination resistance.

---
## 📋 Before You Start
1. Set runtime to **T4 GPU**: `Runtime → Change runtime type → T4 GPU`
2. Upload your `qwen2.5-7b-instruct-q4_k_m.gguf` to Google Drive
3. Default assumed path: `MyDrive/models/qwen2.5-7b-instruct-q4_k_m.gguf`
4. Run cells **top to bottom**

---
## ⚙️ Section 1 — Environment Setup

In [ ]:
# 1A — Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
print('✅ GPU ready!' if 'T4' in result.stdout or 'Tesla' in result.stdout else '❌ No GPU — check runtime type')

In [ ]:
# 1B — Install all dependencies (~3-5 min)
print('Installing lm-evaluation-harness...')
!pip install lm-eval -q
print('Installing llama-cpp-python with CUDA...')
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --upgrade --force-reinstall --no-cache-dir -q
print('Installing supporting libraries...')
!pip install huggingface_hub transformers accelerate datasets numpy pandas matplotlib seaborn -q
print('\n✅ All dependencies installed!')

---
## 📂 Section 2 — Load Quantized Model from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

In [ ]:
# ⚠️ EDIT THIS path if your GGUF is stored elsewhere in Drive
import os
QUANTIZED_MODEL_PATH = "/content/drive/MyDrive/models/qwen2.5-7b-instruct-q4_k_m.gguf"

if os.path.exists(QUANTIZED_MODEL_PATH):
    size_gb = os.path.getsize(QUANTIZED_MODEL_PATH) / 1e9
    print(f'✅ Found: {QUANTIZED_MODEL_PATH}  ({size_gb:.2f} GB)')
else:
    print(f'❌ Not found at: {QUANTIZED_MODEL_PATH}')
    models_dir = "/content/drive/MyDrive/models"
    if os.path.exists(models_dir):
        print('Files in MyDrive/models:', os.listdir(models_dir))

---
## 📥 Section 3 — Download Original Qwen2.5-7B from HuggingFace

In [ ]:
from huggingface_hub import snapshot_download
ORIGINAL_MODEL_DIR = "/content/models/qwen-original"
os.makedirs(ORIGINAL_MODEL_DIR, exist_ok=True)
print('Downloading Qwen2.5-7B-Instruct BF16 (~14GB, ~3-5 min)...')
snapshot_download(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    local_dir=ORIGINAL_MODEL_DIR,
    ignore_patterns=["*.pt", "original/*"]
)
print(f'\n✅ Downloaded to: {ORIGINAL_MODEL_DIR}')

---
## 📊 Section 4 — Perplexity Comparison
**Lower = better.** Direct measure of language quality loss from quantization.

In [ ]:
# 4A — Perplexity: Quantized (GGUF on GPU)
from llama_cpp import Llama
from datasets import load_dataset
import numpy as np

def calc_ppl_gguf(model_path, num_samples=100):
    llm = Llama(model_path=model_path, n_ctx=512, n_gpu_layers=-1, verbose=False)
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    nlls, processed = [], 0
    for sample in dataset:
        if processed >= num_samples: break
        text = sample["text"].strip()
        if len(text) < 100: continue
        if len(llm.tokenize(text.encode())) < 20: continue
        out = llm(text[:1000], max_tokens=1, logprobs=32, echo=True)
        lps = [v for t in out["choices"][0]["logprobs"]["top_logprobs"] for v in t.values() if v is not None]
        if lps:
            nlls.append(-np.mean(lps))
            processed += 1
        if processed % 25 == 0: print(f'  {processed}/{num_samples}...')
    return float(np.exp(np.mean(nlls)))

print('Computing quantized model perplexity...')
ppl_quantized = calc_ppl_gguf(QUANTIZED_MODEL_PATH)
print(f'\n✅ Quantized Perplexity: {ppl_quantized:.4f}')

In [ ]:
# 4B — Perplexity: Original (HF BF16)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def calc_ppl_hf(model_dir, num_samples=100):
    tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_dir, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    model.eval()
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    nlls, processed = [], 0
    with torch.no_grad():
        for sample in dataset:
            if processed >= num_samples: break
            text = sample["text"].strip()
            if len(text) < 100: continue
            enc = tokenizer(text[:1000], return_tensors="pt").to("cuda")
            if enc.input_ids.shape[1] < 20: continue
            nlls.append(model(**enc, labels=enc.input_ids).loss.item())
            processed += 1
            if processed % 25 == 0: print(f'  {processed}/{num_samples}...')
    del model
    torch.cuda.empty_cache()
    return float(np.exp(np.mean(nlls)))

print('Computing original model perplexity...')
ppl_original = calc_ppl_hf(ORIGINAL_MODEL_DIR)
print(f'\n✅ Original Perplexity: {ppl_original:.4f}')

In [ ]:
# 4C — Perplexity Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

BG        = '#0f1117'
CARD      = '#1a1d27'
GRID      = '#2e3250'
COL_ORIG  = '#4f8ef7'
COL_QUANT = '#f7934f'
TEXT      = '#dce1f0'
SUBTEXT   = '#aab0c6'

ppl_increase = ((ppl_quantized - ppl_original) / ppl_original) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor(BG)
for ax in axes:
    ax.set_facecolor(CARD)
    for spine in ax.spines.values(): spine.set_edgecolor(GRID)

# Bar: absolute perplexity
ax1 = axes[0]
bars = ax1.bar(['Original\n(BF16)', 'Quantized\n(Q4_K_M)'],
               [ppl_original, ppl_quantized],
               color=[COL_ORIG, COL_QUANT], width=0.45, zorder=3)
ax1.set_title('Perplexity Score (lower = better)', color=TEXT, fontsize=13, pad=12)
ax1.set_ylabel('Perplexity', color=SUBTEXT)
ax1.tick_params(colors=SUBTEXT)
ax1.grid(axis='y', color=GRID, linestyle='--', zorder=0)
y_min, y_max = min(ppl_original, ppl_quantized)*0.95, max(ppl_original, ppl_quantized)*1.06
ax1.set_ylim(y_min, y_max)
for bar, val in zip(bars, [ppl_original, ppl_quantized]):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+(y_max-y_min)*0.01,
             f'{val:.3f}', ha='center', va='bottom', color=TEXT, fontsize=11, fontweight='bold')

# Gauge: % increase
ax2 = axes[1]
ax2.set_xlim(0, 10); ax2.set_ylim(0, 10); ax2.axis('off')
for (x0, x1, lbl, col) in [(1,3.5,'< 5%\nExcellent','#2ecc71'),(3.5,6.5,'5–10%\nAcceptable','#f39c12'),(6.5,9,'>10%\nDegraded','#e74c3c')]:
    ax2.barh(5, x1-x0, left=x0, height=1.6, color=col, alpha=0.25, zorder=1)
    ax2.text((x0+x1)/2, 4.1, lbl, ha='center', va='top', color=col, fontsize=8.5, fontweight='bold')
marker_x = 1 + (min(ppl_increase, 15)/15)*8
marker_col = '#2ecc71' if ppl_increase < 5 else ('#f39c12' if ppl_increase < 10 else '#e74c3c')
ax2.annotate('', xy=(marker_x,5.8), xytext=(marker_x,7.2),
             arrowprops=dict(arrowstyle='->', color=marker_col, lw=2.5))
ax2.text(marker_x, 7.5, f'{ppl_increase:+.2f}%', ha='center', color=marker_col, fontsize=14, fontweight='bold')
ax2.text(5, 2.8, 'Perplexity Increase from Quantization', ha='center', color=SUBTEXT, fontsize=10)
ax2.set_title('Quality Loss Gauge', color=TEXT, fontsize=13, pad=12)

plt.tight_layout(pad=2)
plt.savefig('/content/perplexity_chart.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Original: {ppl_original:.4f}  |  Quantized: {ppl_quantized:.4f}  |  Δ {ppl_increase:+.2f}%')

---
## 🏆 Section 5 — lm-eval Benchmarks
⏱️ ~20–30 min per model with `--limit 200`

In [ ]:
# 5A — Benchmarks: Quantized Model
os.makedirs('/content/results/quantized', exist_ok=True)
print('Running lm-eval on Q4_K_M... (~20-30 min)')
!lm_eval \
  --model gguf \
  --model_args pretrained={QUANTIZED_MODEL_PATH},n_gpu_layers=-1 \
  --tasks mmlu,hellaswag,truthfulqa_mc1,arc_challenge,gsm8k \
  --num_fewshot 0 \
  --limit 200 \
  --output_path /content/results/quantized \
  --batch_size 1
print('\n✅ Quantized benchmarks complete!')

In [ ]:
# 5B — Benchmarks: Original Model
os.makedirs('/content/results/original', exist_ok=True)
print('Running lm-eval on original BF16... (~20-30 min)')
!lm_eval \
  --model hf \
  --model_args pretrained={ORIGINAL_MODEL_DIR},dtype=bfloat16 \
  --tasks mmlu,hellaswag,truthfulqa_mc1,arc_challenge,gsm8k \
  --num_fewshot 0 \
  --limit 200 \
  --output_path /content/results/original \
  --batch_size 4
print('\n✅ Original benchmarks complete!')

---
## 📈 Section 6 — Results Table + All Visualizations

In [ ]:
# 6A — Parse results
import json, glob
import pandas as pd
import warnings; warnings.filterwarnings('ignore')

def load_results(path):
    files = glob.glob(os.path.join(path,'**','*.json'), recursive=True)
    latest = max(files, key=os.path.getmtime)
    with open(latest) as f: return json.load(f)

TASK_LABELS = {
    'mmlu':           ('MMLU',          'General Knowledge'),
    'hellaswag':      ('HellaSwag',     'Common Sense'),
    'truthfulqa_mc1': ('TruthfulQA',    'Hallucination'),
    'arc_challenge':  ('ARC Challenge', 'Reasoning'),
    'gsm8k':          ('GSM8K',         'Math'),
}
METRIC_KEYS = ['acc,none','acc_norm,none','exact_match,strict-match']

def get_score(data, task):
    r = data.get('results',{}).get(task,{})
    for k in METRIC_KEYS:
        if k in r: return r[k]*100
    return None

q_data = load_results('/content/results/quantized')
o_data = load_results('/content/results/original')

rows = []
for tk,(label,cat) in TASK_LABELS.items():
    o_s = get_score(o_data, tk)
    q_s = get_score(q_data, tk)
    if o_s is None or q_s is None: continue
    delta = q_s - o_s
    verdict = ('✅ Negligible' if abs(delta)<1 else '✅ Good' if abs(delta)<3
               else '⚠️  Acceptable' if abs(delta)<5 else '❌ Significant')
    rows.append({'Task':label,'Category':cat,'Original_BF16':o_s,'Q4_K_M':q_s,'Delta':delta,'Verdict':verdict})

df = pd.DataFrame(rows)
avg_drop = df['Delta'].abs().mean()
overall = ('✅ EXCELLENT — virtually indistinguishable from original' if avg_drop<1.5
           else '✅ GOOD — minor loss, safe for enterprise use' if avg_drop<3
           else '⚠️  ACCEPTABLE — noticeable but manageable' if avg_drop<5
           else '❌ SIGNIFICANT — consider Q5_K_M or Q6_K')

disp = df.copy()
disp['Original BF16'] = disp['Original_BF16'].map('{:.1f}%'.format)
disp['Q4_K_M Score']  = disp['Q4_K_M'].map('{:.1f}%'.format)
disp['Delta']         = disp['Delta'].map('{:+.1f}%'.format)
print(disp[['Task','Category','Original BF16','Q4_K_M Score','Delta','Verdict']].to_string(index=False))
print(f'\nOVERALL: {overall}  |  Avg delta: {avg_drop:.2f}%')

In [ ]:
# 6B — Chart 1: Grouped Bar (score per task, both models side by side)
import matplotlib.pyplot as plt
import numpy as np

BG='#0f1117'; CARD='#1a1d27'; GRID='#2e3250'
COL_ORIG='#4f8ef7'; COL_QUANT='#f7934f'; TEXT='#dce1f0'; SUBTEXT='#aab0c6'

tasks       = df['Task'].tolist()
orig_scores = df['Original_BF16'].tolist()
quant_scores= df['Q4_K_M'].tolist()
x = np.arange(len(tasks)); w = 0.35

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(BG); ax.set_facecolor(CARD)
for s in ax.spines.values(): s.set_edgecolor(GRID)

b1 = ax.bar(x-w/2, orig_scores,  w, label='Original (BF16)',    color=COL_ORIG,  zorder=3)
b2 = ax.bar(x+w/2, quant_scores, w, label='Quantized (Q4_K_M)', color=COL_QUANT, zorder=3)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f'{bar.get_height():.1f}', ha='center', va='bottom', color=TEXT, fontsize=8.5, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(tasks, color=TEXT, fontsize=11)
ax.set_ylabel('Accuracy (%)', color=SUBTEXT, fontsize=11)
ax.set_ylim(0, max(orig_scores+quant_scores)*1.13)
ax.tick_params(colors=SUBTEXT)
ax.grid(axis='y', color=GRID, linestyle='--', zorder=0)
ax.set_title('Benchmark Scores: Original BF16 vs Q4_K_M', color=TEXT, fontsize=14, pad=14)
ax.legend(facecolor=CARD, edgecolor=GRID, labelcolor=TEXT, fontsize=10)

plt.tight_layout()
plt.savefig('/content/grouped_bar.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# 6C — Chart 2: Delta Bar (quality loss per task with threshold lines)
import matplotlib.patches as mpatches

deltas     = df['Delta'].tolist()
bar_colors = ['#2ecc71' if abs(d)<1 else '#f39c12' if abs(d)<3 else '#e67e22' if abs(d)<5 else '#e74c3c' for d in deltas]

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor(BG); ax.set_facecolor(CARD)
for s in ax.spines.values(): s.set_edgecolor(GRID)

bars = ax.bar(tasks, deltas, color=bar_colors, zorder=3, width=0.5)
ax.axhline(0,  color=GRID,    linewidth=1.2, zorder=2)
ax.axhline(-3, color='#f39c12', linewidth=1, linestyle='--', zorder=2, alpha=0.7)
ax.axhline(-5, color='#e74c3c', linewidth=1, linestyle='--', zorder=2, alpha=0.7)
ax.text(len(tasks)-0.5, -3.2, '−3% warning',  color='#f39c12', fontsize=8, ha='right')
ax.text(len(tasks)-0.5, -5.2, '−5% critical', color='#e74c3c', fontsize=8, ha='right')

for bar, val in zip(bars, deltas):
    ypos = val-0.3 if val<0 else val+0.1
    ax.text(bar.get_x()+bar.get_width()/2, ypos, f'{val:+.1f}%',
            ha='center', va='top' if val<0 else 'bottom', color=TEXT, fontsize=9, fontweight='bold')

ax.set_xticklabels(tasks, color=TEXT, fontsize=11)
ax.set_ylabel('Score Delta (%)', color=SUBTEXT, fontsize=11)
ax.tick_params(colors=SUBTEXT)
ax.grid(axis='y', color=GRID, linestyle='--', zorder=0)
ax.set_title('Quality Loss per Task  (Q4_K_M vs Original)', color=TEXT, fontsize=14, pad=14)

legend_patches = [
    mpatches.Patch(color='#2ecc71', label='< 1% — Negligible'),
    mpatches.Patch(color='#f39c12', label='1–3% — Good'),
    mpatches.Patch(color='#e67e22', label='3–5% — Acceptable'),
    mpatches.Patch(color='#e74c3c', label='> 5% — Significant'),
]
ax.legend(handles=legend_patches, facecolor=CARD, edgecolor=GRID, labelcolor=TEXT, fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig('/content/delta_bar.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# 6D — Chart 3: Radar / Spider Chart (capability shape)
labels  = df['Task'].tolist()
orig_v  = df['Original_BF16'].tolist()
quant_v = df['Q4_K_M'].tolist()
N       = len(labels)
angles  = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]
orig_v  += orig_v[:1]
quant_v += quant_v[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
fig.patch.set_facecolor(BG); ax.set_facecolor(CARD)
ax.spines['polar'].set_edgecolor(GRID)

ax.plot(angles, orig_v,  color=COL_ORIG,  linewidth=2.2, label='Original (BF16)')
ax.fill(angles, orig_v,  color=COL_ORIG,  alpha=0.15)
ax.plot(angles, quant_v, color=COL_QUANT, linewidth=2.2, label='Quantized (Q4_K_M)')
ax.fill(angles, quant_v, color=COL_QUANT, alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, color=TEXT, fontsize=11)
ax.set_rlabel_position(30)
ax.yaxis.set_tick_params(labelcolor=SUBTEXT, labelsize=8)
ax.grid(color=GRID, linestyle='--', linewidth=0.7)
ax.legend(loc='upper right', bbox_to_anchor=(1.3,1.1),
          facecolor=CARD, edgecolor=GRID, labelcolor=TEXT, fontsize=10)
ax.set_title('Capability Radar: Original vs Quantized', color=TEXT, fontsize=13, pad=20)

plt.tight_layout()
plt.savefig('/content/radar_chart.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# 6E — Chart 4: Summary Scorecard (one-glance overview of all results)
fig, ax = plt.subplots(figsize=(13, 6.5))
fig.patch.set_facecolor(BG); ax.set_facecolor(BG); ax.axis('off')

ax.text(0.5, 0.97, 'Quantization Quality Scorecard',
        ha='center', va='top', transform=ax.transAxes,
        fontsize=17, fontweight='bold', color=TEXT)
ax.text(0.5, 0.90, 'Qwen2.5-7B-Instruct  |  Original BF16  →  Q4_K_M (llama.cpp)',
        ha='center', va='top', transform=ax.transAxes, fontsize=10, color=SUBTEXT)

card_w, card_h = 0.17, 0.46
starts_x = [0.025, 0.215, 0.405, 0.595, 0.785]
card_y   = 0.27

for i, row in df.iterrows():
    d   = row['Delta']
    col = '#2ecc71' if abs(d)<1 else '#f39c12' if abs(d)<3 else '#e67e22' if abs(d)<5 else '#e74c3c'
    x0  = starts_x[i]
    cx  = x0 + card_w/2

    rect = mpatches.FancyBboxPatch((x0, card_y), card_w, card_h,
        boxstyle='round,pad=0.01', linewidth=1.5,
        edgecolor=col, facecolor=CARD, transform=ax.transAxes, zorder=2)
    ax.add_patch(rect)

    accent = mpatches.FancyBboxPatch((x0, card_y+card_h-0.055), card_w, 0.055,
        boxstyle='round,pad=0.005', linewidth=0,
        facecolor=col, alpha=0.35, transform=ax.transAxes, zorder=3)
    ax.add_patch(accent)

    ax.text(cx, card_y+card_h-0.02, row['Task'],     ha='center', va='top', transform=ax.transAxes, fontsize=10,   fontweight='bold', color=TEXT,    zorder=4)
    ax.text(cx, card_y+card_h-0.09, row['Category'], ha='center', va='top', transform=ax.transAxes, fontsize=7.5,  color=SUBTEXT, zorder=4)
    ax.text(cx, card_y+0.25, f'{d:+.1f}%',           ha='center', va='center', transform=ax.transAxes, fontsize=21, fontweight='bold', color=col, zorder=4)
    ax.text(cx, card_y+0.17, 'score change',          ha='center', va='center', transform=ax.transAxes, fontsize=7,  color=SUBTEXT, zorder=4)
    ax.text(cx, card_y+0.10, f'BF16:   {row["Original_BF16"]:.1f}%', ha='center', va='center', transform=ax.transAxes, fontsize=8, color=COL_ORIG,  zorder=4)
    ax.text(cx, card_y+0.03, f'Q4_K_M: {row["Q4_K_M"]:.1f}%',       ha='center', va='center', transform=ax.transAxes, fontsize=8, color=COL_QUANT, zorder=4)

# Perplexity pill
try:
    ppl_col = '#2ecc71' if ppl_increase<5 else ('#f39c12' if ppl_increase<10 else '#e74c3c')
    ax.text(0.5, 0.21,
            f'Perplexity  |  Original: {ppl_original:.2f}   →   Q4_K_M: {ppl_quantized:.2f}   |   Δ {ppl_increase:+.2f}%',
            ha='center', va='center', transform=ax.transAxes, fontsize=10, color=ppl_col,
            bbox=dict(boxstyle='round,pad=0.4', facecolor=CARD, edgecolor=ppl_col, linewidth=1.2))
except NameError:
    pass

# Overall verdict
ax.text(0.5, 0.08, f'OVERALL: {overall}',
        ha='center', va='center', transform=ax.transAxes, fontsize=11, fontweight='bold', color=TEXT,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#12151f', edgecolor=GRID, linewidth=1.2))

plt.tight_layout()
plt.savefig('/content/scorecard.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# 6F — Save everything to Google Drive
import shutil, json
from datetime import datetime

OUT_DIR = '/content/drive/MyDrive/models/benchmark_results'
os.makedirs(OUT_DIR, exist_ok=True)

for fname in ['perplexity_chart.png','grouped_bar.png','delta_bar.png','radar_chart.png','scorecard.png']:
    src = f'/content/{fname}'
    if os.path.exists(src): shutil.copy(src, os.path.join(OUT_DIR, fname))

report = {
    'generated_at': datetime.now().isoformat(),
    'model_quantized': QUANTIZED_MODEL_PATH,
    'model_original': 'Qwen/Qwen2.5-7B-Instruct (BF16)',
    'perplexity': {
        'original':     ppl_original  if 'ppl_original'  in dir() else None,
        'quantized':    ppl_quantized if 'ppl_quantized' in dir() else None,
        'pct_increase': ppl_increase  if 'ppl_increase'  in dir() else None,
    },
    'benchmark_results': df.to_dict(orient='records') if 'df' in dir() else [],
    'overall_verdict':   overall   if 'overall'   in dir() else None,
    'avg_delta_pct':     float(avg_drop) if 'avg_drop' in dir() else None,
}
with open(os.path.join(OUT_DIR,'benchmark_report.json'),'w') as f:
    json.dump(report, f, indent=2)

print(f'✅ Saved to Google Drive: {OUT_DIR}')
print('Files:')
for f in os.listdir(OUT_DIR): print(f'  - {f}')

---
## 🔍 Section 7 — Optional: Qualitative Spot Checks
Side-by-side responses on enterprise-relevant prompts. Good gut-check that numbers match real-world feel.

In [ ]:
from llama_cpp import Llama
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

PROMPTS = [
    {'label': '💼 Financial Reasoning',
     'prompt': "A company's debt-to-equity ratio rose from 0.6 to 1.8 over three years while revenue grew 5% annually. Is this a cause for concern? Give a brief analysis."},
    {'label': '📧 Email Drafting',
     'prompt': 'Draft a short professional email informing a client their Q3 financial report will be delayed by one week due to a data reconciliation issue.'},
    {'label': '🧠 Hallucination Test',
     'prompt': 'What was the closing price of Apple stock on January 15, 2019? Answer only if certain, otherwise say you do not know.'},
]

print('Loading Q4_K_M...')
llm_q = Llama(model_path=QUANTIZED_MODEL_PATH, n_ctx=1024, n_gpu_layers=-1, verbose=False)
print('Loading original BF16...')
tok = AutoTokenizer.from_pretrained(ORIGINAL_MODEL_DIR, trust_remote_code=True)
m   = AutoModelForCausalLM.from_pretrained(ORIGINAL_MODEL_DIR, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True)
pipe = pipeline('text-generation', model=m, tokenizer=tok, max_new_tokens=200)

for item in PROMPTS:
    print('\n' + '━'*70)
    print(item['label'])
    print(f'Prompt: {item["prompt"]}\n')
    q_resp = llm_q.create_chat_completion(messages=[{'role':'user','content':item['prompt']}], max_tokens=200)
    o_resp = pipe(item['prompt'], return_full_text=False)
    print(f'🔷 Original (BF16):\n{o_resp[0]["generated_text"]}')
    print(f'\n🔶 Quantized (Q4_K_M):\n{q_resp["choices"][0]["message"]["content"]}')